# IE–IATA Datathon 2025 – Objective 1: SAF uptake & emissions (EU27)

This notebook reconstructs our Objective 1 pipeline:

- **Scope**: flights departing EU27 airports (EU27 aggregation, UK/CH excluded).
- **Years**: 2025–2050 (model), with official output 2026–2050.
- **Scenarios**: S0 = Market-driven BAU (no mandates), S1 = Policy-accelerated (ReFuelEU + 5–10 pp from 2030).

It uses only public data (Eurostat, ReFuelEU) and clearly documented assumptions as required by the challenge brief.



In [ ]:
import pandas as pd
import numpy as np
import math

pd.set_option("display.max_columns", 10)
pd.set_option("display.precision", 3)



In [ ]:
# Load data from GitHub (replace <your-branch> with your actual branch name, e.g., 'main' or 'feat/step4-obj1-output')
# Alternative: upload CSVs directly to Colab if GitHub URLs don't work

BRANCH = "main"  # Update this to your branch name after merging
BASE_URL = f"https://raw.githubusercontent.com/hadimoumeni/iata-datathon/{BRANCH}/data/clean"

traffic_url = f"{BASE_URL}/traffic_projection__eu27__annual__2025-2050.csv"
blend_url   = f"{BASE_URL}/saf_blend_targets__eu27__annual__2025-2050.csv"
base_url    = f"{BASE_URL}/jet_fuel_baseline__eu27__annual__1990-LATEST_mt.csv"

traffic = pd.read_csv(traffic_url)
blend   = pd.read_csv(blend_url)
base    = pd.read_csv(base_url)

print("Traffic projection:")
print(traffic.head())
print("\nSAF blend targets:")
print(blend.head())
print("\nJet fuel baseline (last 5 years):")
print(base.tail())



## Key assumptions (documented for Obj1)

Based on the datathon brief and supporting references:

- **Baseline EU27 fuel demand**: ~38–39 Mt in 2024/2025 → we target 38.5 Mt in 2025 for calibration.
- **Fossil jet emission factor**: 3.16 tCO₂ per tonne of fuel.
- **Central SAF life-cycle CO₂ reduction**: 75% vs fossil jet fuel (within the 70–80% range suggested by the brief).
- **Traffic index**: represents EU27 flight activity; 2025 = 100.

All values are stored in `references/assumptions.yaml` in the repo.



In [ ]:
# ---- Assumptions (documented in references/assumptions.yaml) --------------
EF_TCO2_PER_T = 3.16     # tCO2 per tonne fuel
SAF_LCA_REDUCTION_PCT = 75.0  # SAF lifecycle reduction vs fossil (%)
RECOMMENDED_BASELINE_2025_MT = 38.5  # Mt total fuel in 2025

# ---- Prepare data ----------------------------------------------------------
# Ensure integer years
for df in (traffic, blend, base):
    df['year'] = df['year'].astype(int)

# Determine baseline_2025 (Mt)
if (base['year'] == 2025).any():
    baseline_2025_mt = float(base.loc[base['year']==2025, 'jet_fuel_mt'].iloc[0])
    print(f"Using 2025 baseline from data: {baseline_2025_mt:.2f} Mt")
else:
    baseline_2025_mt = RECOMMENDED_BASELINE_2025_MT
    print(f"Using recommended baseline: {baseline_2025_mt} Mt")

# Scale traffic index to Mt so that 2025 hits the baseline
idx_2025 = float(traffic.loc[traffic['year']==2025, 'traffic_index_2025=100'].iloc[0])
if idx_2025 == 0 or math.isnan(idx_2025):
    raise ValueError("traffic index for 2025 is missing or zero.")

combined = traffic.merge(blend, on='year', how='left')

# Fill missing traffic index values (forward projection)
# Simple assumption: flat traffic if only 2025 exists
traffic_known = combined[combined['traffic_index_2025=100'].notna()].copy()
if len(traffic_known) > 1:
    # If we have historical data, use last known growth rate
    last_idx = traffic_known['traffic_index_2025=100'].iloc[-1]
    prev_idx = traffic_known['traffic_index_2025=100'].iloc[-2]
    growth_rate = (last_idx / prev_idx) ** (1.0 / (traffic_known['year'].iloc[-1] - traffic_known['year'].iloc[-2]))
    # Project forward
    for year in range(2026, 2051):
        if pd.isna(combined.loc[combined['year']==year, 'traffic_index_2025=100'].values[0]):
            years_ahead = year - traffic_known['year'].iloc[-1]
            combined.loc[combined['year']==year, 'traffic_index_2025=100'] = last_idx * (growth_rate ** years_ahead)
else:
    # If only 2025 exists, assume flat traffic (index stays at 100)
    combined['traffic_index_2025=100'] = combined['traffic_index_2025=100'].fillna(100.0)

combined['total_fuel_mt'] = baseline_2025_mt * (combined['traffic_index_2025=100'] / idx_2025)

# Helper to compute metrics for a scenario column ('S0' or 'S1')
def compute_for_scenario(df, scen_col, scen_id, ef=EF_TCO2_PER_T, saf_red=SAF_LCA_REDUCTION_PCT):
    out = df[['year','total_fuel_mt',scen_col]].copy()
    out.rename(columns={scen_col:'saf_share_pct'}, inplace=True)
    # shares are in %, convert to fraction
    frac = out['saf_share_pct'] / 100.0
    out['saf_mt'] = out['total_fuel_mt'] * frac
    out['jet_mt'] = out['total_fuel_mt'] - out['saf_mt']
    # CO2 generated = jet*EF + saf*EF*(1 - LCA_reduction)
    out['co2_generated_mt'] = out['jet_mt']*ef + out['saf_mt']*ef*(1.0 - saf_red/100.0)
    # Avoided CO2 = baseline(0% SAF) - generated
    out['co2_avoided_mt'] = out['total_fuel_mt']*ef - out['co2_generated_mt']
    # Assemble judge schema; filter to 2026–2050 per spec
    o = out[(out['year']>=2026) & (out['year']<=2050)][
        ['year','total_fuel_mt','saf_share_pct','co2_generated_mt','co2_avoided_mt']
    ].copy()
    o.insert(1, 'Scenario', scen_id)
    o.rename(columns={
        'year':'Year',
        'total_fuel_mt':'Total_Fuel',
        'saf_share_pct':'SAF_Share',
        'co2_generated_mt':'CO2_Emissions',
        'co2_avoided_mt':'Avoided_CO2'
    }, inplace=True)
    # Optional rounding for readability (not required by spec)
    for c in ['Total_Fuel','SAF_Share','CO2_Emissions','Avoided_CO2']:
        o[c] = o[c].astype(float).round(3)
    return o

# Compute for both scenarios
res0 = compute_for_scenario(combined, 'S0', 0)
res1 = compute_for_scenario(combined, 'S1', 1)
final = pd.concat([res0, res1], ignore_index=True).sort_values(['Year','Scenario'])

# Validate schema & ranges
required_cols = ['Year','Scenario','Total_Fuel','SAF_Share','CO2_Emissions','Avoided_CO2']
assert list(final.columns) == required_cols, f"Schema mismatch: {final.columns}"
assert final['Year'].min()==2026 and final['Year'].max()==2050, "Year range must be 2026–2050"

print("\n✓ Computation complete!")
print(f"Total rows: {len(final)} (25 years × 2 scenarios)")



In [ ]:
# Required table structure check (mandatory per brief)
print("=" * 60)
print("FINAL OUTPUT TABLE STRUCTURE")
print("=" * 60)

print("\n1. First 10 rows:")
print(final.head(10))

print("\n2. Column names and data types:")
print(final.dtypes)

print("\n3. Column list:")
print(list(final.columns))

print("\n4. Sample row (2026, Scenario 0 - BAU):")
print(final[(final["Year"]==2026) & (final["Scenario"]==0)])

print("\n5. Sample row (2050, Scenario 1 - Policy):")
print(final[(final["Year"]==2050) & (final["Scenario"]==1)])

print("\n6. Year range check:")
print(f"Min year: {final['Year'].min()}, Max year: {final['Year'].max()}")
print(f"Scenarios: {sorted(final['Scenario'].unique())}")

print("\n7. Last 5 rows:")
print(final.tail(5))



In [ ]:
# Optional: Quick visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 8))

# Plot 1: Total Fuel and SAF Share over time
ax1 = axes[0]
for scen in [0, 1]:
    subset = final[final['Scenario']==scen]
    ax1.plot(subset['Year'], subset['Total_Fuel'], 
             marker='o', label=f'S{scen} Total Fuel (Mt)', linewidth=2)
    ax1_twin = ax1.twinx()
    ax1_twin.plot(subset['Year'], subset['SAF_Share'], 
                  marker='s', linestyle='--', label=f'S{scen} SAF Share (%)', alpha=0.7)

ax1.set_xlabel('Year')
ax1.set_ylabel('Total Fuel (Mt)', color='blue')
ax1_twin.set_ylabel('SAF Share (%)', color='green')
ax1.set_title('Total Fuel Demand and SAF Blend Share by Scenario')
ax1.legend(loc='upper left')
ax1_twin.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Plot 2: CO2 Emissions and Avoided CO2
ax2 = axes[1]
for scen in [0, 1]:
    subset = final[final['Scenario']==scen]
    ax2.plot(subset['Year'], subset['CO2_Emissions'], 
             marker='o', label=f'S{scen} CO2 Emissions (Mt)', linewidth=2)
    ax2.plot(subset['Year'], subset['Avoided_CO2'], 
             marker='s', linestyle='--', label=f'S{scen} Avoided CO2 (Mt)', alpha=0.7)

ax2.set_xlabel('Year')
ax2.set_ylabel('CO2 (Mt)')
ax2.set_title('CO2 Emissions and Avoided CO2 by Scenario')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSummary statistics:")
print(final.groupby('Scenario')[['Total_Fuel', 'SAF_Share', 'CO2_Emissions', 'Avoided_CO2']].agg(['mean', 'min', 'max']))

